# NAIC 2026 — N-ATLaS Phase 0 Feasibility & Multilingual Verification

This notebook verifies genuine **N-ATLaS** (`NCAIR1/N-ATLaS`, Llama 3 8B) inference in **Google Colab** on a T4 GPU (or A100/L4).

### Objectives:
1. Verify GPU acceleration & VRAM.
2. Authenticate with Hugging Face (`HF_TOKEN`) for the gated N-ATLaS repository.
3. Load `NCAIR1/N-ATLaS` using 4-bit NormalFloat (`bitsandbytes` NF4).
4. Benchmark model loading time, VRAM usage, and generation latency.
5. Evaluate baseline responses across **English**, **Hausa**, **Yoruba**, and **Igbo**.
6. Export a structured verification artifact (`natlas_phase0_results.json`) for `docs/FEASIBILITY.md`.

## 1. Verify Colab GPU Environment
Ensure you have a GPU runtime enabled in Colab (**Runtime > Change runtime type > T4 GPU**).

In [1]:
import torch

print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Assigned GPU:", torch.cuda.get_device_name(0))
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


GPU Available: False


In [1]:
import sys
import platform

print("Python:", sys.version)
print("Platform:", platform.platform())

Python: 3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]
Platform: Windows-11-10.0.26200-SP0


In [7]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

PyTorch: 2.11.0+cpu
CUDA available: False
CUDA version: None


In [10]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    raise SystemError("Please enable a GPU under Runtime -> Change runtime type!")

## 2. Install Required Python Dependencies
We install `transformers`, `accelerate`, `bitsandbytes`, `peft`, and `sentencepiece`.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes peft datasets sentencepiece

## 3. Hugging Face Authentication
`NCAIR1/N-ATLaS` is a **gated repository**. You must accept the model conditions on Hugging Face (https://huggingface.co/NCAIR1/N-ATLaS) and provide your Hugging Face User Access Token below.

In [ ]:
from huggingface_hub import login
from google.colab import userdata

# You can either save your token in Colab Secrets as 'HF_TOKEN' or enter it interactively
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("Successfully logged in via Colab Secret HF_TOKEN!")
except Exception:
    print("Please enter your Hugging Face Access Token:")
    login()

## 4. Load N-ATLaS with 4-bit Quantization (bitsandbytes NF4)
4-bit quantization reduces memory consumption from ~16GB to ~5.5GB VRAM, allowing smooth inference and fine-tuning on a standard 15-16GB T4 GPU.

In [ ]:
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "NCAIR1/N-ATLaS"

print("Configuring 4-bit quantization (NF4)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

print(f"Loading tokenizer for {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

start_time = time.time()
print(f"Loading model weights for {model_id}...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)
load_duration = time.time() - start_time

allocated_vram = torch.cuda.memory_allocated() / (1024**3)
reserved_vram = torch.cuda.memory_reserved() / (1024**3)

print(f"\n Model loaded successfully in {load_duration:.2f} seconds!")
print(f"VRAM Allocated: {allocated_vram:.2f} GB")
print(f"VRAM Reserved : {reserved_vram:.2f} GB")

## 5. Run Multilingual Generation Benchmark
We evaluate genuine inference performance on 4 core languages (English, Hausa, Yoruba, Igbo).

In [ ]:
import json
from datetime import datetime

test_cases = [
    {
        "language": "English",
        "domain": "Health Reporting",
        "prompt": "A health worker in Kaduna reports: 'Our clinic received 35 children with acute malaria symptoms this week, but our artemisinin-based combination therapy (ACT) supply is down to 4 packs.' Extract the key issues into a structured summary: Location, Condition, Cases, Current Supply, Action Required."
    },
    {
        "language": "Hausa",
        "domain": "Health Awareness",
        "prompt": "Menene matakan kariya mafi mahimmanci da al'umma za su dauka domin dakile yaduwar zazzabin cizon sauro a lokacin damina?"
    },
    {
        "language": "Yoruba",
        "domain": "Community Health",
        "prompt": "Kí ni àwọn àmì pàtàkì tí a fi ń mọ àrùn ibà ní àwọn ọmọdé, àti kí ni ó yẹ kí àwọn òbí ṣe ní kíákíá?"
    },
    {
        "language": "Igbo",
        "domain": "Facility Intelligence",
        "prompt": "Kọwaa ihe kpatara o ji dị mkpa ka ụlọ ọrụ ahụike dị n'ime obodo nwee ọgwụ mgbochi ọrịa mgbe niile maka ụmụaka."
    }
]

benchmark_results = {
    "benchmark_date": datetime.now().isoformat(),
    "model": model_id,
    "gpu": torch.cuda.get_device_name(0),
    "vram_total_gb": round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2),
    "vram_allocated_gb": round(allocated_vram, 2),
    "load_duration_seconds": round(load_duration, 2),
    "generations": []
}

print("=" * 65)
print("N-ATLaS MULTILINGUAL INFERENCE BENCHMARK")
print("=" * 65)

for case in test_cases:
    lang = case["language"]
    prompt = case["prompt"]
    print(f"\n>>> Language: {lang} ({case['domain']})")
    print(f"Prompt: {prompt}\n")
    
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    input_len = inputs.input_ids.shape[1]
    
    torch.cuda.synchronize()
    t_start = time.time()
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=180,
            temperature=0.6,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    torch.cuda.synchronize()
    t_elapsed = time.time() - t_start
    
    output_len = outputs.shape[1] - input_len
    tok_per_sec = output_len / t_elapsed if t_elapsed > 0 else 0
    response_text = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    
    print(f"Response ({output_len} tokens | {t_elapsed:.2f}s | {tok_per_sec:.1f} tok/s):\n")
    print(response_text)
    print("-" * 65)
    
    benchmark_results["generations"].append({
        "language": lang,
        "domain": case["domain"],
        "prompt": prompt,
        "response": response_text,
        "input_tokens": int(input_len),
        "output_tokens": int(output_len),
        "latency_seconds": round(t_elapsed, 2),
        "tokens_per_second": round(tok_per_sec, 1)
    })

## 6. Save & Download Results Artifact
Download the generated JSON file so we can save it to `evaluation/results/natlas_phase0_results.json` in your local project.

In [ ]:
from google.colab import files

results_filename = "natlas_phase0_results.json"
with open(results_filename, "w", encoding="utf-8") as f:
    json.dump(benchmark_results, f, indent=2, ensure_ascii=False)

print(f"Saved {results_filename} successfully!")
files.download(results_filename)